# Tutorial 4, Q2a: factory B on its own

*Original script: `5 Source Files/Python/tutorial_q2a_factoryB.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Create a Concrete Model
model = ConcreteModel()

In [ ]:
# Define integer variables
model.x = Var(range(2), domain=NonNegativeIntegers)

In [ ]:
# Define objective function coefficients
profit = [10, 15]

In [ ]:
# Define coefficients for constraints
grind = [5, 3]
polish = [5, 6]
raw = [4, 4]

In [ ]:
# Objective function
model.obj = Objective(expr=sum(profit[i] * model.x[i] for i in model.x), sense=maximize)

In [ ]:
# Define the inequality constraints
model.grinding_cap = Constraint(expr=sum(grind[i] * model.x[i] for i in model.x) <= 60)
model.polishing_cap = Constraint(expr=sum(polish[i] * model.x[i] for i in model.x) <= 75)
model.raw_cap = Constraint(expr=sum(raw[i] * model.x[i] for i in model.x) <= 45)

In [ ]:
# Solve the optimization problem
solver = SolverFactory('appsi_highs')  # You can use a different solver here
solver.solve(model, tee=True)

In [ ]:
# Print the model
model.pprint()

In [ ]:
# Print the results
model.display()

In [ ]:
# Calculate surpluses
surplus_grind = 60 - sum(grind[i] * value(model.x[i]) for i in model.x)
surplus_polish = 75 - sum(polish[i] * value(model.x[i]) for i in model.x)
surplus_raw = 45 - sum(raw[i] * value(model.x[i]) for i in model.x)

print("Surplus in grinding capacity:", surplus_grind)
print("Surplus in polishing capacity:", surplus_polish)
print("Surplus in raw material capacity:", surplus_raw)